# SAVA-CLIP experiments on Kaggle

Before running: **Settings → Accelerator = GPU T4 x2**, **Internet = On**, and add your uploaded code dataset with **Add Input**.

Then click **Save Version → Save & Run All (Commit)**. Everything runs in the background (about 4–6 h). When it finishes, download `paper_outputs.zip` from the **Output** tab.

## Step 1 — Check the GPU

In [ ]:
!nvidia-smi

## Step 2 — Copy the code from the uploaded dataset

In [ ]:
import glob, os, shutil
hits = glob.glob('/kaggle/input/**/code/run_all.py', recursive=True)
assert hits, 'Code not found: click "Add Input" and add your SAVA-CLIP dataset.'
src = os.path.dirname(os.path.dirname(hits[0]))
dst = '/kaggle/working/SAVA-CLIP'
if not os.path.exists(dst):
    shutil.copytree(src, dst)
print('Code copied to', dst)
os.chdir(dst + '/code')
print(os.listdir('.'))

## Step 3 — Install libraries

In [ ]:
!pip install -q 'transformers>=4.45,<5' accelerate scikit-learn
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## Step 4 — Download the datasets (PlantVillage colour images + PlantDoc)

In [ ]:
%cd /kaggle/working
!test -d PlantVillage-Dataset || (git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset && cd PlantVillage-Dataset && git sparse-checkout set raw/color)
!test -d PlantDoc-Dataset || git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset
!echo PlantVillage images: $(find PlantVillage-Dataset/raw/color -type f | wc -l)   "(expected 54305)"
!echo PlantDoc images: $(find PlantDoc-Dataset/train PlantDoc-Dataset/test -type f | wc -l)

In [ ]:
%cd /kaggle/working/SAVA-CLIP/code
PV = '/kaggle/working/PlantVillage-Dataset/raw/color'
PD = '/kaggle/working/PlantDoc-Dataset'

## Step 5 — Main experiments: zero-shot, few-shot, ablation, full data (~1.5–2 h)

In [ ]:
!python run_all.py --pv_root {PV} --pd_root {PD} --out results 2>&1 | grep -v Warning | tee log_run_all.txt | tail -40

## Step 6 — ResNet-50 baseline (~1 h)

In [ ]:
!python train_cnn.py --arch resnet50 --pv_root {PV} --pd_root {PD} --out results 2>&1 | grep -v Warning | tee log_resnet.txt | tail -15

## Step 7 — ViT-B/16 baseline (~1.5–2 h)

In [ ]:
!python train_cnn.py --arch vit_b_16 --pv_root {PV} --pd_root {PD} --out results 2>&1 | grep -v Warning | tee log_vit.txt | tail -15

## Step 8 — Qwen2-VL zero-shot baseline (~1 h; optional)
If this step fails or takes too long, the paper can drop the Qwen2-VL row.

In [ ]:
!python generative_vlm.py --pv_root {PV} --pd_root {PD} --out results --n_pv 950 2>&1 | grep -v Warning | tee log_genvlm.txt | tail -10

## Step 9 — Write the numbers into results.tex and draw the result figures

In [ ]:
!python write_tex.py --out results
!python make_figures.py --out results --heatmaps ../fig/samples/Apple___Apple_scab.jpg ../fig/samples/Tomato___Leaf_Mold.jpg ../fig/samples/Potato___Late_blight.jpg ../fig/samples/Grape___Black_rot.jpg 2>&1 | grep -v Warning | tail -3

## Step 10 — Package everything for download

In [ ]:
import zipfile, glob, os
os.chdir('/kaggle/working/SAVA-CLIP')
files = (['results.tex'] + glob.glob('fig/fig_*.pdf') + glob.glob('code/results/*.json')
         + glob.glob('code/results/*.npz') + glob.glob('code/results/sava_full.pt') + glob.glob('code/log_*.txt'))
with zipfile.ZipFile('/kaggle/working/paper_outputs.zip', 'w') as z:
    for f in files:
        if os.path.exists(f):
            z.write(f)
print('\n'.join(files))
print('\nDownload /kaggle/working/paper_outputs.zip from the Output tab.')

## Quick look at the main numbers

In [ ]:
import json
R = json.load(open('/kaggle/working/SAVA-CLIP/code/results/results.json'))
for m in ['resnet','vit','zsname','zssape','genvlm','lp','clipad','tipf','sava']:
    if m in R:
        print(f"{m:8s}  PlantVillage acc {R[m]['pv']['acc']:6.2f}   PlantDoc acc {R[m]['pd']['acc']:6.2f}")